# Prerequisite 02 · NumPy：shape 是第一等公民

[上一章：Python](Prerequisite01_Python.ipynb) · [下一章：Matplotlib](Prerequisite03_Matplotlib.ipynb)

**目标：**熟练处理 state、batch 和 action 维度，理解广播与高级索引。RL 中大量“算法错误”其实是 shape 悄悄错了。


## 1. 数组、shape、dtype

一个状态可能是 `(features,)`，一批状态通常是 `(batch, features)`，Q 网络输出通常是 `(batch, actions)`。


In [1]:
import numpy as np

states = np.array([[0.1, 0.2, -0.3], [0.0, 0.4, 0.5]], dtype=np.float32)
weights = np.array([[1.0, -1.0], [0.5, 0.2], [-0.3, 0.8]], dtype=np.float32)
q_values = states @ weights
print("states:", states.shape, states.dtype)
print("Q(s,a):", q_values.shape, "\n", q_values)
assert q_values.shape == (2, 2)


states: (2, 3) float32
Q(s,a): (2, 2) 
 [[ 0.29000002 -0.3       ]
 [ 0.05        0.48000002]]


## 2. axis 与动作索引

`axis=1` 表示在每个样本的动作维上归约。训练 DQN 时，需要从每行 Q 值中取出实际执行动作对应的一项。


In [2]:
actions = np.array([1, 0])
batch_index = np.arange(len(actions))
chosen_q = q_values[batch_index, actions]
greedy_actions = q_values.argmax(axis=1)

print("chosen Q:", chosen_q)
print("greedy actions:", greedy_actions)
print("max next Q:", q_values.max(axis=1))


chosen Q: [-0.3   0.05]
greedy actions: [0 1]
max next Q: [0.29000002 0.48000002]


## 3. 广播：方便，也危险

`(B,A)` 加 `(B,1)` 会把每个样本的标量复制到所有动作；加 `(B,)` 通常无法表达你想要的语义。先在纸上写 shape，再写代码。


In [3]:
advantages = np.array([[1.0, 2.0], [-1.0, 3.0]])
values = np.array([[10.0], [20.0]])
dueling_q = values + advantages - advantages.mean(axis=1, keepdims=True)
print(dueling_q)
assert np.allclose(dueling_q.mean(axis=1), values[:, 0])


[[ 9.5 10.5]
 [18.  22. ]]


## 4. 随机数与可复现

推荐使用 `np.random.default_rng(seed)`，并把生成器显式传入实验边界。相同 seed 是调试工具，不代表一次运行就足以得出结论。


In [4]:
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)
sample_a = rng_a.normal(size=4)
sample_b = rng_b.normal(size=4)
print(sample_a)
assert np.allclose(sample_a, sample_b)


[ 0.30471708 -1.03998411  0.7504512   0.94056472]


## 练习与过关标准

**练习：**给定 `q.shape == (4,3)` 和 `actions == [2,0,1,2]`，不用循环取出四个已选动作的 Q 值。

<details><summary>答案</summary>`q[np.arange(4), actions]`。结果 shape 应为 `(4,)`。</details>

**过关：**看到数组运算能说出输入、输出 shape；会使用 `reshape`、布尔 mask、`axis`、矩阵乘法和 action indexing。

**官方选读：**[NumPy absolute basics](https://numpy.org/doc/stable/user/absolute_beginners)；需要更系统时读 [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)。
